In [1]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Use EE initialization from luma_ge
import ee 
import luma_ge

# Autheticate using service account (json file)

service_account_path = '../auth/ee-epstm2024.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")


Service account initialization failed: Caller does not have required permission to use project ee-epstm2024. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=ee-epstm2024 and then retry. Propagation of the new permission may take a few minutes.


Earth Engine initialized with service account successfully!
Initialized: True
Authenticated: True
Project: projects/ee-epstm2024/assets/Reference_data_sumsel_test


In [3]:
# Collect satellite images

import geemap
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image

In [4]:
# AOI using agil's data

import pandas as pd
indo_regency = ee.FeatureCollection('projects/ee-agilakbar/assets/Indonesian_Regency')
#check Regency List
regency = indo_regency.aggregate_array("WADMKK").getInfo()
#print(pd.DataFrame(regency, columns=["WADMKK"]))
#province = indo_regency.aggregate_array("WADMPR").getInfo()
#Example for Pagar Alam
regency_name = "Kota Pagar Alam"
#Filter the FeatureCollection, used it for AOI
aoi = indo_regency.filter(ee.Filter.eq("WADMKK", regency_name)).geometry()

In [5]:
#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2017-01-01'
end = '2017-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
#Add the data to the map
Map = geemap.Map()
Map.addLayer(mosaic_landsat, l8_sr_visparam, 'L8 SR Mosaic')
Map.addLayer(median_landsat, l8_sr_visparam, 'L8 SR Median')
Map.addLayer(landsat_data, l8_sr_visparam, 'L8 SR Image Collection')
# set center of the map in the area of interest
Map.centerObject(aoi, 7)

#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()
#visualize the thermal bands and multispectral bands
Map.addLayer(median_thermal, thermal_vis, "Thermal Bands")
#Map

2026-03-13 18:13:27,084 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-03-13 18:13:27,087 - final_Image - INFO - final_Image creation initialized.
2026-03-13 18:13:27,089 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-03-13 18:13:27,090 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-03-13 18:13:27,091 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-03-13 18:13:27,091 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-03-13 18:13:27,093 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-03-13 18:13:27,095 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-03-13 18:13:27,611 - final_Image - INFO - Creating quality mosaic from 5 images using NDVI as quality metric
2026-03-13 18:13:27,614 - final_Image - INFO - Quality mosaic created covering AOI with best available pixels
2026

In [6]:
import ee


class SyntheticModularTrainingData:
    """
    Create synthetic modular training dataset for testing primitive layers.
    Assumes Earth Engine already initialized.
    """

    def __init__(self, aoi, n_points=200, seed=42):

        if not isinstance(aoi, ee.Geometry):
            raise ValueError("AOI must be ee.Geometry")

        self.aoi = aoi
        self.n_points = n_points
        self.seed = seed

    # -------------------------------------------------
    # Generate random points
    # -------------------------------------------------

    def _generate_points(self):

        pts = ee.FeatureCollection.randomPoints(
            region=self.aoi,
            points=self.n_points,
            seed=self.seed
        )

        return pts

    # -------------------------------------------------
    # Add random columns
    # -------------------------------------------------

    def _add_random_columns(self, fc):

        fc = fc.randomColumn("r1", self.seed)
        fc = fc.randomColumn("r2", self.seed + 1)
        fc = fc.randomColumn("r3", self.seed + 2)
        fc = fc.randomColumn("r4", self.seed + 3)
        fc = fc.randomColumn("r5", self.seed + 4)

        return fc

    # -------------------------------------------------
    # Assign labels
    # -------------------------------------------------

    def _assign_labels(self, feature):

        tree = ee.Number(feature.get("r1")).gt(0.5)
        built = ee.Number(feature.get("r2")).gt(0.7)
        soil = ee.Number(feature.get("r3")).gt(0.6)
        water = ee.Number(feature.get("r4")).gt(0.8)

        tree_temp = ee.Algorithms.If(
            tree,
            ee.Number(feature.get("r5")).gt(0.5),
            0
        )

        return feature.set({
            "tree_presence": tree,
            "builtupsurface_presence": built,
            "baresoil_presence": soil,
            "waterbody_presence": water,
            "tree_temporal_variation": tree_temp
        })

    # -------------------------------------------------
    # Public
    # -------------------------------------------------

    def create(self):

        pts = self._generate_points()

        pts = self._add_random_columns(pts)

        labeled = pts.map(self._assign_labels)

        labeled = labeled.select([
            "tree_presence",
            "builtupsurface_presence",
            "baresoil_presence",
            "waterbody_presence",
            "tree_temporal_variation"
        ])

        return labeled

In [9]:
synthetic = SyntheticModularTrainingData(
    aoi=aoi,
    n_points=300
)

training_fc = synthetic.create()

preview = training_fc.limit(10).getInfo()

for f in preview["features"]:
    print(f["properties"])

inside_test = training_fc.map(
    lambda f: f.set(
        "inside",
        ee.Geometry(aoi).contains(f.geometry())
    )
)

print(
    inside_test.aggregate_histogram("inside").getInfo()
)

{'baresoil_presence': 0, 'builtupsurface_presence': 1, 'tree_presence': 0, 'tree_temporal_variation': 0, 'waterbody_presence': 0}
{'baresoil_presence': 0, 'builtupsurface_presence': 1, 'tree_presence': 0, 'tree_temporal_variation': 0, 'waterbody_presence': 0}
{'baresoil_presence': 1, 'builtupsurface_presence': 0, 'tree_presence': 0, 'tree_temporal_variation': 0, 'waterbody_presence': 1}
{'baresoil_presence': 0, 'builtupsurface_presence': 1, 'tree_presence': 1, 'tree_temporal_variation': 1, 'waterbody_presence': 0}
{'baresoil_presence': 1, 'builtupsurface_presence': 0, 'tree_presence': 0, 'tree_temporal_variation': 0, 'waterbody_presence': 1}
{'baresoil_presence': 0, 'builtupsurface_presence': 0, 'tree_presence': 1, 'tree_temporal_variation': 0, 'waterbody_presence': 0}
{'baresoil_presence': 0, 'builtupsurface_presence': 0, 'tree_presence': 1, 'tree_temporal_variation': 1, 'waterbody_presence': 0}
{'baresoil_presence': 1, 'builtupsurface_presence': 0, 'tree_presence': 0, 'tree_temporal_

In [10]:
from luma_ge.classification import FeatureExtraction
from luma_ge.classification import Generate_LULC

class PrimitiveLayerTrainer:

    def __init__(self, image, roi):

        self.image = image
        self.roi = roi

        self.elements = [
            "tree_presence",
            "builtupsurface_presence",
            "baresoil_presence",
            "waterbody_presence",
            "tree_temporal_variation"
        ]

        self.fe = FeatureExtraction()
        self.clf = Generate_LULC()

    # ------------------------------

    def train_one(
        self,
        element,
        split_ratio=0.7,
        pixel_size=10,
        ntrees=50,
        seed=0
    ):

        train_pix, test_pix = self.fe.random_split(
            image=self.image,
            roi=self.roi,
            class_property=element,
            split_ratio=split_ratio,
            pixel_size=pixel_size
        )

        primitive = self.clf.hard_classification(
            training_data=train_pix,
            class_property=element,
            image=self.image,
            ntrees=ntrees,
            seed=seed
        )

        return primitive.rename(element)

    # ------------------------------

    def train_all(self):

        layers = {}

        for el in self.elements:

            print("Training:", el)

            layers[el] = self.train_one(el)

        return layers

In [12]:
# Feature Extraction

# predictor image (from your previous step)
image = stacked_landsat

# modular training data
roi = training_fc

trainer = PrimitiveLayerTrainer(
    image=image,
    roi=roi
)

primitive_layers = trainer.train_all()

print(primitive_layers.keys())

Training: tree_presence
Single Random Split Training Pixel Size: 187
Single Random Split Testing Pixel Size: 94
Training: builtupsurface_presence
Single Random Split Training Pixel Size: 187
Single Random Split Testing Pixel Size: 94
Training: baresoil_presence
Single Random Split Training Pixel Size: 187
Single Random Split Testing Pixel Size: 94
Training: waterbody_presence
Single Random Split Training Pixel Size: 187
Single Random Split Testing Pixel Size: 94
Training: tree_temporal_variation
Single Random Split Training Pixel Size: 187
Single Random Split Testing Pixel Size: 94
dict_keys(['tree_presence', 'builtupsurface_presence', 'baresoil_presence', 'waterbody_presence', 'tree_temporal_variation'])


In [13]:
import geemap

m = geemap.Map()

m.centerObject(aoi, 12)

m.addLayer(aoi, {}, "AOI")

m.addLayer(
    training_fc,
    {"color": "red"},
    "Training points"
)

m.addLayer(
    primitive_layers["tree_presence"],
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "tree_presence"
)
m.addLayer(
    primitive_layers["builtupsurface_presence"],
    {"min": 0, "max": 1, "palette": ["white", "red"]},
    "builtupsurface_presence"
)
m.addLayer(
    primitive_layers["baresoil_presence"],
    {"min": 0, "max": 1, "palette": ["white", "brown"]},
    "baresoil_presence"
)

m

Map(center=[-4.1175978213298485, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparen…

In [14]:
# after training each primitive
for key in primitive_layers:
    primitive_layers[key] = primitive_layers[key].rename(key)

primitive_image = ee.Image.cat(list(primitive_layers.values()))

In [15]:
# Create datacube of the primitive layers

primitive_image = ee.Image.cat([
    primitive_layers["tree_presence"],
    primitive_layers["builtupsurface_presence"],
    primitive_layers["baresoil_presence"],
    primitive_layers["waterbody_presence"],
    primitive_layers["tree_temporal_variation"]
])

print(primitive_image.bandNames().getInfo())

['tree_presence', 'builtupsurface_presence', 'baresoil_presence', 'waterbody_presence', 'tree_temporal_variation']


In [16]:
# Define classification schemes

import pandas as pd

scheme_data = [

    # ---------- SCHEME 1 ----------

    ["scheme1", 1, "Forest",
     "tree_presence == 1"],

    ["scheme1", 2, "Built",
     "builtupsurface_presence == 1"],

    ["scheme1", 3, "Water",
     "waterbody_presence == 1"],


    # ---------- SCHEME 2 ----------

    ["scheme2", 1, "StableForest",
     "tree_presence == 1 AND tree_temporal_variation == 0"],

    ["scheme2", 2, "DynamicVeg",
     "tree_presence == 1 AND tree_temporal_variation == 1"],

    ["scheme2", 3, "NonVeg",
     "tree_presence == 0"]

]

df_rules = pd.DataFrame(
    scheme_data,
    columns=[
        "scheme",
        "class_id",
        "class_name",
        "rule"
    ]
)

print(df_rules)

    scheme  class_id    class_name  \
0  scheme1         1        Forest   
1  scheme1         2         Built   
2  scheme1         3         Water   
3  scheme2         1  StableForest   
4  scheme2         2    DynamicVeg   
5  scheme2         3        NonVeg   

                                                rule  
0                                 tree_presence == 1  
1                       builtupsurface_presence == 1  
2                            waterbody_presence == 1  
3  tree_presence == 1 AND tree_temporal_variation...  
4  tree_presence == 1 AND tree_temporal_variation...  
5                                 tree_presence == 0  


In [22]:
# Ruleset classifier

import ee


class RuleSetClassifier:

    def __init__(self, primitive_image, rules_df, aoi):

        self.image = primitive_image
        self.df = rules_df
        self.aoi = aoi

    # -----------------------------

    def _rule_to_expression(self, rule):

        expr = rule.replace("AND", "&&")
        expr = expr.replace("OR", "||")

        return expr

    # -----------------------------

    def classify_scheme(self, scheme_name):

        subset = self.df[self.df["scheme"] == scheme_name]

        result = ee.Image(0).rename("dummy")

        # safer: get band names once
        band_names = self.image.bandNames().getInfo()

        band_dict = {}
        for name in band_names:
            band_dict[name] = self.image.select(name)

        for _, row in subset.iterrows():

            class_id = int(row["class_id"])
            rule = row["rule"]

            expr = self._rule_to_expression(rule)

            mask = ee.Image().expression(
                expr,
                band_dict
            )

            # ensure mask is boolean
            mask = mask.eq(1)

            result = result.where(mask, class_id)
        
        result = result.clip(self.aoi)

        return result.rename(scheme_name)

In [23]:
classifier = RuleSetClassifier(
    primitive_image=primitive_image,
    rules_df=df_rules,
    aoi=aoi
)

map1 = classifier.classify_scheme("scheme1")

map2 = classifier.classify_scheme("scheme2")

print(map1.bandNames().getInfo())  # ['scheme1']
print(map2.bandNames().getInfo())  # ['scheme2']

['scheme1']
['scheme2']


In [24]:
# Visualize

import geemap

m = geemap.Map()
m.centerObject(aoi, 12)

vis1 = {
    "min": 1,
    "max": 3,
    "palette": ["006400", "a52a2a", "0000ff"]
}

vis2 = {
    "min": 1,
    "max": 3,
    "palette": ["228b22", "7fff00", "d3d3d3"]
}

m.addLayer(map1.clip(aoi), vis1, "Scheme1")
m.addLayer(map2.clip(aoi), vis2, "Scheme2")
m.addLayer(primitive_image, {}, "Primitive stack", False)

legend1 = {
    "Forest": "006400",
    "Built": "a52a2a",
    "Water": "0000ff",
}

m.add_legend(
    title="Scheme1",
    legend_dict=legend1
)

legend2 = {
    "StableForest": "228b22",
    "DynamicVeg": "7fff00",
    "NonVeg": "d3d3d3",
}

m.add_legend(
    title="Scheme2",
    legend_dict=legend2
)

m

Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

In [25]:
def class_hist(img):

    stats = img.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi,
        scale=30,
        maxPixels=1e9
    )

    return stats.getInfo()

print("Scheme1:", class_hist(map1))
print("Scheme2:", class_hist(map2))

Scheme1: {'scheme1': {'0': 298920.9647058822, '1': 302005.7098039202, '2': 91616.93333333336, '3': 9357.313725490194}}
Scheme2: {'scheme2': {'0': 52662.02352941176, '1': 341984.30980392016, '2': 21977.92156862744, '3': 285276.6666666667}}


In [ ]:
diff = map1.neq(map2)

m.addLayer(
    diff.clip(aoi),
    {"min":0,"max":1,"palette":["white","black"]},
    "Difference"
)

m

# If everything black → schemes identical
# If random noise → primitives wrong
# If meaningful pattern → good

Map(bottom=2145335.0, center=[-4.106311062929824, 103.22581860642481], controls=(WidgetControl(options=['posit…